# 101 · Ledger: The Source of Truth

At the heart of `EarlySign` is the **Ledger**. Instead of storing the *current state* of an experiment in a traditional database table, we store the **history of events**. This approach, known as **Event Sourcing**, allows us to reconstruct the state of an analysis at any point in time with full auditability and lineage.

## Why Event Sourcing for Statistics?
1. **Reproducibility**: You can exactly replicate a decision by replaying the events.
2. **Auditability**: Every statistical fact (Z-score, p-value) is linked to a specific set of raw observations.
3. **Flexibility**: You can change your analysis logic and "back-fill" results from the same raw data without losing history.

In this tutorial, we will learn how to:
- Initialize a `Ledger`.
- Write raw observations and analytical designs.
- Query the ledger using `Ibis` to derive statistical states.

## 1. Setup

We start by creating an in-memory database (DuckDB) and initializing the `Ledger`. A Ledger is a table with a specific schema designed for performance: `timestamp`, `ledger_id` (primary identifier), `uuid`, `type`, `payload` (JSON), `attributes` (JSON), and `metadata` (JSON).

In [1]:
import ibis
from pydantic import BaseModel

from earlysign.core.ledger import Ledger

# Create an in-memory connection
con = ibis.connect("duckdb://:memory:")

# Initialize a Ledger named 'tutorial_events' with a specific ledger_id for performance/clustering
ledger = Ledger(con, "tutorial_events", ledger_id="101_demo")

# Ensure the table exists with the correct schema (and clustering columns if on BigQuery)
ledger.ensure()

print("Ledger initialized.")

Ledger initialized.


## 2. Writing to the Ledger

Every "write" is an immutable event. We use types (classes) to distinguish between different kinds of events.

In [2]:
class Design(BaseModel):
    test_name: str
    alpha: float
    metric: str


class Observation(BaseModel):
    look_index: int
    arms: list[dict]


# 1. Record the Design of our experiment
ledger.insert(
    data=Design(
        test_name="Binomial A/B",
        alpha=0.05,
        metric="conversion_rate",
    )
)

# 2. Record a batch of Observations
ledger.insert(
    data=Observation(
        look_index=1,
        arms=[
            {"name": "control", "n": 500, "success": 55},
            {"name": "treatment", "n": 500, "success": 72},
        ],
    )
)

print("Design and Observations recorded.")

Design and Observations recorded.


## 3. Reading and Deriving State

The power of the Ledger is that we can use `Ibis` (a portable dataframe-like API) to query events. The `ledger.t` property gives us an Ibis expression pointing to the table.

In [3]:
# View the raw records (as a Pandas DataFrame)
df = ledger.t.execute()
display(df)

,uuid,type,payload,attributes,timestamp,ledger_id,metadata
0,9009310f107d4f2cb9c9842cd1ab5787,Design,"{'test_name': 'Binomial A/B', 'alpha': 0.05, '...",{},2026-02-11 05:24:30.361403+00:00,101_demo,{'pkg_version': 'earlysign==0.0.0'}
1,fdab6a7caec041678849a7c0c839f628,Observation,"{'look_index': 1, 'arms': [{'name': 'control',...",{},2026-02-11 05:24:30.567763+00:00,101_demo,{'pkg_version': 'earlysign==0.0.0'}


## 4. Deriving Statistics on the Fly

Instead of reading a pre-computed "conversion_rate" column, we calculate it from the raw events. Because we use Ibis, this same code work identically on DuckDB, BigQuery, or Snowflake.

In [4]:
def calculate_conversion(t):
    # Filter for observations related to our experiment
    # The type is automatically derived from the class name (Observation)
    obs = t.filter(t.type == "Observation")

    # In a real scenario, you'd use Ibis expressions to unnest JSON payloads.
    # For this low-level tutorial, we'll execute and use Python logic for clarity.
    pdf = obs.execute()

    for _, row in pdf.iterrows():
        p = row["payload"]
        # Handle both string and dict payloads depending on backend/cache
        import json

        if isinstance(p, str):
            p = json.loads(p)

        print(f"Look {p['look_index']}:")
        for arm in p["arms"]:
            rate = arm["success"] / arm["n"]
            print(f"  - {arm['name']}: {rate:.1%} ({arm['success']}/{arm['n']})")


calculate_conversion(ledger.t)

Look 1:
  - control: 11.0% (55/500)
  - treatment: 14.4% (72/500)


## 5. Summary

- **Source of Truth**: The Ledger is the definitive record of what happened.
- **ledger_id**: We use it for high-performance grouping and clustering in production warehouses.
- **Attributes**: We continue to use them for secondary, dynamic JSON-based labeling.
- **Derivation**: State is reconstructed by querying history, not by updating rows.

In the next tutorial, we will see how the **Framework layer** automates this read/write process using sessions and type-safe projectors.